# Generación de datos

A continuación vamoas a generar datos, para el futuro análisis, se ha generar 4 conjuntos de datos, que serán transformados y subidos a diferentes origenes.
Tenemos:

1. Información general --> csv
2. Información de bienestar --> xlsx
3. Economía, ingresos --> BD, mysql
4. Economía, egresos --> BD, postgres

Generación nombres estudiantes y ciudad de origen

In [1]:
!pip install faker unidecode -q

from faker import Faker
from unidecode import unidecode
import pandas as pd

fake = Faker(['es_ES', 'es_MX', 'es_CO', 'es_AR'])  # español variado
Faker.seed(42)

N = 5000

seen = set()
rows = []
while len(rows) < N:
    full = fake.name()
    key = unidecode(full).strip().lower()
    if key not in seen:
        seen.add(key)
        partes = full.split()
        nombre = " ".join(partes[:-1]) if len(partes) > 1 else partes[0]
        apellido = partes[-1] if len(partes) > 1 else ""
        rows.append((nombre, apellido))

names = pd.DataFrame(rows, columns=['nombre', 'apellido'])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.7 MB/s eta 0:00:00


In [2]:
ciudades=pd.read_excel('Ciudades_geocodificadas.xlsx')
ciudades= ciudades.head(150).copy()

# latitud y longitud de Quito
quito = ciudades[ciudades["Ciudad"].astype(str).str.strip().str.lower() == "quito"].iloc[0]
lat_q, lon_q = float(quito["lat"]), float(quito["lon"])

import math
# calculo distancia con latitud y longitud
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

#calculo distancia
ciudades["distancia"] = ciudades.apply(
    lambda r: haversine(lat_q, lon_q, float(r["lat"]), float(r["lon"])),
    axis=1
).round(2)

ciudades = ciudades[ciudades["Ciudad"].astype(str).str.strip().str.lower() != "quito"]
ciudades["Población"] = (
    ciudades["Población"]
    .astype(str)                     # asegúrate de que todo sea texto
    .str.replace(r"\s+", "", regex=True)  # quita todos los espacios
    .str.replace(".", "", regex=False)    # quita puntos si los hay
    .astype(int)                     # convierte a número entero
)

Vamos a asignar las ciudades con probabilidad ajustada, entre más lejos esté menos probabilidad tiene y entre mayor es la población más probabilidad tiene.

In [3]:
dist_norm = ciudades["distancia"] / ciudades["distancia"].max()
pop_norm  = ciudades["Población"] / ciudades["Población"].max()

score = 0.8 * (1 - dist_norm) + 0.3 * pop_norm # más importancia tiene la distancia
score = score.clip(lower=1e-12)
prob = score / score.sum()


In [4]:
import numpy as np

# elegimos las ciudades
np.random.seed(43)
idx_elegidos = np.random.choice(ciudades.index, size=len(names), p=prob.values)

# corregismo nombres
asignadas = ciudades.loc[idx_elegidos, ["Ciudad","Provincia","lon","lat","distancia"]].reset_index(drop=True)
asignadas = asignadas.rename(columns={
    "Ciudad": "ciudad",
    "Provincia": "provincia",
    "lon": "longitud",
    "lat": "latitud",
    "distancia": "distancia"
})

# df de donde sacaremos los otros
df_base = pd.concat([names, asignadas], axis=1)
df_base["distancia"] = df_base["distancia"].round(2)

df_base.head()

,nombre,apellido,ciudad,provincia,longitud,latitud,distancia
0,Raimundo Llopis,Hierro,Latacunga,Cotopaxi,-78.614576,-0.934031,80.19
1,Benjamín Zoé,Trejo,Tosagua,Manabí,-80.257719,-0.775521,203.66
2,Ileana,Antón-Andrés,Sangolquí,Pichincha,-78.448348,-0.328882,14.03
3,Demetrio,Vazquez,Balzar,Guayas,-79.936715,-1.306299,199.16
4,Chita del,Giménez,Jaramijó,Manabí,-80.613458,-0.974388,248.22


Vamos a asignarles una Universidad a cada estudiantes

In [5]:
universidades_publicas_quito = [
    {"nombre": "Universidad Central del Ecuador (UCE)", "estudiantes": 42000},
    {"nombre": "Escuela Politécnica Nacional (EPN)", "estudiantes": 12000},
    {"nombre": "Escuela Politécnica del Ejército (ESPE)", "estudiantes": 15000},
    {"nombre": "Universidad Andina Simón Bolívar (UASB)", "estudiantes": 2500},
    {"nombre": "Instituto de Altos Estudios Nacionales (IAEN)", "estudiantes": 1500},
]
universidades_privadas_quito = [
    {"nombre": "Universidad de las Américas (UDLA)", "estudiantes": 25000},
    {"nombre": "Pontificia Universidad Católica del Ecuador (PUCE)", "estudiantes": 21000},
    {"nombre": "Universidad Internacional del Ecuador (UIDE)", "estudiantes": 18000},
    {"nombre": "Universidad San Francisco de Quito (USFQ)", "estudiantes": 10000},
    {"nombre": "Universidad Politécnica Salesiana (UPS)", "estudiantes": 9000},
    {"nombre": "Universidad Tecnológica Equinoccial (UTE)", "estudiantes": 8000},
    {"nombre": "Universidad Tecnológica Indoamérica (UTI)", "estudiantes": 7000},
]
pub = pd.DataFrame(universidades_publicas_quito)
pub["tipo"] = "Pública"
priv = pd.DataFrame(universidades_privadas_quito)
priv["tipo"] = "Privada"


In [6]:
# normalizamos la población, entre más población más oportunidad de ir a una privada
ciudades["pob_norm"] = ciudades["Población"] / ciudades["Población"].max()
poblacion_map = ciudades.set_index("Ciudad")["pob_norm"].to_dict() # ciudad: probabilidad privada

np.random.seed(42)

tipos = []
for ciudad in df_base["ciudad"]:
    prob_priv = poblacion_map.get(ciudad, 0.3)
    prob_priv = min(prob_priv + 0.18, 0.95)
    tipo = np.random.choice(["Privada", "Pública"], p=[prob_priv, 1 - prob_priv])
    tipos.append(tipo)

base = df_base.copy()
df_base["tipo_universidad"] = tipos

def elegir_universidad(tipo):
    if tipo == "Privada":
        probs = priv["estudiantes"] / priv["estudiantes"].sum()
        return np.random.choice(priv["nombre"], p=probs)
    else:
        probs = pub["estudiantes"] / pub["estudiantes"].sum()
        return np.random.choice(pub["nombre"], p=probs)

df_base["universidad"] = df_base["tipo_universidad"].apply(elegir_universidad)
df_general = df_base.copy()
df_base = base

df_general.head()

,nombre,apellido,ciudad,provincia,longitud,latitud,distancia,tipo_universidad,universidad
0,Raimundo Llopis,Hierro,Latacunga,Cotopaxi,-78.614576,-0.934031,80.19,Pública,Universidad Central del Ecuador (UCE)
1,Benjamín Zoé,Trejo,Tosagua,Manabí,-80.257719,-0.775521,203.66,Pública,Universidad Central del Ecuador (UCE)
2,Ileana,Antón-Andrés,Sangolquí,Pichincha,-78.448348,-0.328882,14.03,Pública,Escuela Politécnica del Ejército (ESPE)
3,Demetrio,Vazquez,Balzar,Guayas,-79.936715,-1.306299,199.16,Pública,Universidad Central del Ecuador (UCE)
4,Chita del,Giménez,Jaramijó,Manabí,-80.613458,-0.974388,248.22,Privada,Universidad Tecnológica Equinoccial (UTE)


En el df_bienestar vamos a colocar el promedio académico, le damos una pequeña ventaja a ciudades importantes, porque tienen mejores recursos

In [7]:
cap_por_prov = {
    "Azuay": "Cuenca",
    "Bolívar": "Guaranda",
    "Cañar": "Azogues",
    "Carchi": "Tulcán",
    "Cotopaxi": "Latacunga",
    "Chimborazo": "Riobamba",
    "El Oro": "Machala",
    "Esmeraldas": "Esmeraldas",
    "Galapagos": "Puerto Baquerizo Moreno",
    "Guayas": "Guayaquil",
    "Imbabura": "Ibarra",
    "Loja": "Loja",
    "Los Rios": "Babahoyo",
    "Manabí": "Portoviejo",
    "Morona Santiago": "Macas",
    "Napo": "Tena",
    "Orellana": "Puerto Francisco de Orellana",
    "Pastaza": "Puyo",
    "Pichincha": "Quito",
    "Santa Elena": "Santa Elena",
    "Santo Domingo de los Tsáchilas": "Santo Domingo",
    "Sucumbíos": "Nueva Loja",
    "Tungurahua": "Ambato",
    "Zamora Chinchipe": "Zamora",
}


In [8]:
def norm(s):
    return unidecode(str(s)).strip().lower()
cap_map_norm = {norm(k): norm(v) for k, v in cap_por_prov.items()}
prov_norm = df_base["provincia"].astype(str).map(norm)
city_norm = df_base["ciudad"].astype(str).map(norm)

is_capital = [
    city_norm.iat[i] == cap_map_norm.get(prov_norm.iat[i], "")
    for i in range(len(df_base))
]
is_capital = np.array(is_capital, dtype=int)

In [9]:
rng = np.random.default_rng(42)

bonus_capital = 0.20    # ventaja leve para capital
sigma = 0.50            # dispersión de notas
cap_share = is_capital.mean()
base_mu = 8.4 - bonus_capital * cap_share  # compensar para mantener media global ≈ 8.4

mu_i = base_mu + is_capital * bonus_capital
raw = rng.normal(loc=mu_i, scale=sigma, size=len(df_base))
prom = np.clip(raw, 6.0, 9.8)  # evita extremos 0/10 y outliers

df_bienestar = df_base.copy()
df_bienestar["promedio_académico"] = np.round(prom, 2)

Ahora vamos a colocar el número de vaijes que hace hacia su ciudad

In [10]:

np.random.seed(42)
d_norm = df_bienestar["distancia"] / df_bienestar["distancia"].max()
viajes = 10 - 4 * d_norm + np.random.normal(0, 1, len(df_bienestar)) # quitamos en base a distancia y aumentamos con disto. normal

viajes = np.clip(viajes, 5, 15)

df_bienestar["viajes_origen"] = np.round(viajes).astype(int)


En el df_economia_ingresos vamos a poner datos sobre estudiantes becados o no

In [11]:
df_merged = df_general[["nombre", "apellido", "tipo_universidad", "universidad"]].copy()
df_merged = df_merged.merge(
    df_bienestar[["nombre", "apellido", "promedio_académico"]],
    on=["nombre", "apellido"],
    how="left"
)

# probabilidad en base a nota 0-1
p_norm = (df_merged["promedio_académico"] - df_merged["promedio_académico"].min()) / (df_merged["promedio_académico"].max() - df_merged["promedio_académico"].min())

ventaja_privada = 1.2
priv_mask = df_merged["tipo_universidad"] == "Privada"
peso_tipo = np.where(priv_mask, ventaja_privada, 1.0) # priv:1.3, pub:1.0

np.random.seed(42)
prop_becados = 0.1
base_prob = 0.01

prob_beca = base_prob + p_norm * peso_tipo
prob_beca *= prop_becados / prob_beca.mean()

prob_beca = np.clip(prob_beca, 0, 1)

df_merged["becado"] = np.random.rand(len(df_merged)) < prob_beca
df_merged["becado"] = df_merged["becado"].astype(int)  # 1=becado, 0=no

df_economia_ingresos = df_base.copy()
df_economia_ingresos["becado"] = df_merged["becado"]

Ahora pondremos el tipo de ingreso

In [12]:
np.random.seed(42)

tipos_variable = ["Trabajo", "Crédito educativo", "Otro"]
prob_trabajo = 0.25
prob_credito = 0.15
prob_otro = 0.15

registros = []

for i, row in df_economia_ingresos.iterrows():
    ingresos = []

    if np.random.rand() < prob_trabajo:
        ingresos.append("Trabajo")
    if np.random.rand() < prob_credito:
        ingresos.append("Crédito educativo")
    if np.random.rand() < prob_otro:
        ingresos.append("Otro")

    # solo puede tener máx 2 ingresos de los anteriores
    if len(ingresos) > 2:
        ingresos = list(np.random.choice(ingresos, size=2, replace=False))
    # Si no salió ninguno le ponemos apoyo familiar
    if len(ingresos) == 0:
        ingresos = ["Apoyo familiar"]
    else:
        # si solo salió 1 puede que tenga o no apoyo familiar 50%
        if np.random.rand() < 0.5:
            ingresos.append("Apoyo familiar")

    # Si tiene beca se le agrega
    if row["becado"] == 1:
        ingresos.append("Beca")

    # Una fila por tipo de ingreso
    for ingreso in ingresos:
        new_row = row.copy()
        new_row["tipo_ingreso"] = ingreso
        registros.append(new_row)

df_economia_ingresos = pd.DataFrame(registros).reset_index(drop=True)



In [13]:
df_ingresos = df_economia_ingresos.merge(
    df_general[["nombre", "apellido", "tipo_universidad"]],
    on=["nombre", "apellido"],
    how="left"
)
# Normalizar población
ciudades_tmp = ciudades.copy()
ciudades_tmp["pob_norm"] = ciudades_tmp["Población"] / ciudades_tmp["Población"].max()
pob_map = dict(zip(ciudades_tmp["Ciudad"], ciudades_tmp["pob_norm"]))

# Asignamos pob_norm directamente
df_ingresos["pob_norm"] = df_ingresos["ciudad"].map(pob_map).fillna(0.5)

def calcular_ingreso(row):
    tipo = row["tipo_ingreso"]
    uni = row["tipo_universidad"]
    pob = row["pob_norm"]
    ruido = np.random.normal(1, 0.2)

    if tipo == "Apoyo familiar":
        base = 500 * (0.7 + 0.6 * pob)
        if uni == "Privada":
            base *= 1.4
        ingreso = base * ruido

    elif tipo == "Trabajo":
        ingreso = 250 * ruido

    elif tipo == "Crédito educativo":
        ingreso = 300 * ruido

    elif tipo == "Beca":
        ingreso = (np.random.uniform(370, 500) if uni == "Privada"
                   else np.random.uniform(150, 270)) * ruido

    elif tipo == "Otro":
        ingreso = np.random.uniform(200, 500)

    else:
        ingreso = 0

    return round(max(ingreso, 100), 2)

np.random.seed(42)
df_ingresos["ingreso_cantidad"] = df_ingresos.apply(calcular_ingreso, axis=1)

df_economia_ingresos = df_ingresos.copy()
df_economia_ingresos.drop(columns=["pob_norm", "ciudad", "provincia","longitud", "latitud", "distancia"], inplace=True)


A continuación vamos a crear los registros de gastos

In [14]:
# clculo ingresos totales
df_ingresos_totales = (
    df_economia_ingresos
    .groupby(["nombre", "apellido"], as_index=False)
    .agg({
        "ingreso_cantidad": "sum",
        "becado": "max",
        "tipo_universidad": "first"
    })
)

df_ingresos_totales = df_ingresos_totales.rename(columns={"ingreso_cantidad": "ingreso_total"})
df_base_merged = df_base.merge(
    df_ingresos_totales,
    on=["nombre", "apellido"],
    how="left"
)



In [15]:
categorias = [
    ("Vivienda", 0.30),
    ("Alimentación", 0.25),
    ("Transporte", 0.15),
    ("Educación", 0.10),
    ("Ocio y personales", 0.12),
    ("Otros", 0.08),
]

np.random.seed(42)
cats = [c for c, _ in categorias]
weights = np.array([w for _, w in categorias], dtype=float)

prop_gasto_total = 0.80   # 80% del ingreso total se va a gastos
dirichlet_concentracion = 60

rng = np.random.default_rng(42)

registros = []
for _, row in df_base_merged.iterrows():
    gasto_total = float(row.get("ingreso_total", 0.0)) * prop_gasto_total

    if not np.isfinite(gasto_total) or gasto_total <= 0:
        asignaciones = np.zeros(len(cats))
    else:
        alpha = weights * dirichlet_concentracion
        props = rng.dirichlet(alpha)  # proporciones aleatorias con media=weights
        asignaciones = gasto_total * props

    asignaciones_2d = np.round(asignaciones, 2)
    diff = round(gasto_total - asignaciones_2d.sum(), 2)
    if abs(diff) >= 0.01:
        # ajusta la categoría con mayor asignación
        idx_max = int(np.argmax(asignaciones_2d))
        asignaciones_2d[idx_max] = round(asignaciones_2d[idx_max] + diff, 2)

    # Crear 6 filas (una por categoría)
    for cat, monto in zip(cats, asignaciones_2d):
        new_row = row.copy()
        new_row["categoria_gasto"] = cat
        new_row["gasto_categoria"] = float(monto)
        registros.append(new_row)

df_gastos = pd.DataFrame(registros).reset_index(drop=True)


In [16]:
df_gastos= df_gastos.drop(columns=["provincia", "longitud", "latitud", "distancia", "ingreso_total"])

In [17]:
df_merge = df_ingresos_totales.merge(
    df_bienestar[["nombre", "apellido","promedio_académico"]],
    on=["nombre", "apellido"],
    how="left"
)

In [18]:
prom_norm = df_merge["promedio_académico"] / df_merge["promedio_académico"].max()
ingresos_norm = df_merge["ingreso_total"] / df_merge["ingreso_total"].max()

score = 0.8 * prom_norm + 0.2 * ingresos_norm # más importancia tiene la distancia
score = np.round(score.clip(lower=1e-12)*10+0.5,2)
score = score.clip(upper=10)

df_bienestar["indice_bienestar"] = score

# origen_Bienestar


Eliminacion de campos de longitud y latitud, se realiza el calculo de distancia desde el lugar origen a las respectivas univerdades, para el indice de bienestar se agrega los campos de horas de sueño y el tiempo de traslado minimo hacia la universidad

In [19]:
df_bienestar["origen_bienestar"] = np.where(
    is_capital == 1,
    "Capital de provincia",
    "Ciudad no capital"
)
df_bienestar = df_bienestar.drop(columns=["longitud", "latitud"], errors="ignore")

In [20]:
np.random.seed(42)

n = len(df_bienestar)


dist = np.random.normal(loc=8, scale=4, size=n)
dist = np.clip(dist, 0.1, 20)   # mínimo 100 m, máximo 20 km

mask_cerca = np.random.rand(n) < 0.2
dist[mask_cerca] = np.random.uniform(0.1, 1.0, mask_cerca.sum())

df_bienestar["distancia_universidad"] = dist.round(2)




In [21]:
np.random.seed(42)
horas = np.random.normal(7, 1.2, len(df_bienestar))
df_bienestar["horas_sueño"] = np.clip(horas, 4, 10).round(2)


In [31]:
np.random.seed(42)

dist = df_bienestar["distancia_universidad"].astype(float)
velocidad = np.random.normal(loc=20, scale=5, size=len(df_bienestar))
velocidad = np.clip(velocidad, 5, 23)  # no menos de 5 km/h ni más de 40
tiempo = (dist / velocidad) * 60
ruido = np.random.normal(loc=1, scale=0.2, size=len(df_bienestar))

df_bienestar["tiempo_traslado_min"] = np.clip(tiempo * ruido, 5, 120).round(1)



In [32]:
base = score
base_norm = (base - base.min()) / (base.max() - base.min()) * 10

sleep_z = (df_bienestar["horas_sueño"] - df_bienestar["horas_sueño"].mean()) / df_bienestar["horas_sueño"].std()
sleep_adj = sleep_z * 1.5

trans_z = (df_bienestar["tiempo_traslado_min"].mean() - df_bienestar["tiempo_traslado_min"]) / df_bienestar["tiempo_traslado_min"].std()
trans_adj = trans_z * 3

df_bienestar["indice_bienestar"] = (base_norm + sleep_adj + trans_adj).clip(0, 10).round(2)

In [33]:
df_bienestar

,nombre,apellido,ciudad,provincia,distancia,promedio_académico,viajes_origen,indice_bienestar,origen_bienestar,distancia_universidad,horas_sueño,tiempo_traslado_min
0,Raimundo Llopis,Hierro,Latacunga,Cotopaxi,80.19,8.72,10,8.40,Capital de provincia,0.38,7.60,5.0
1,Benjamín Zoé,Trejo,Tosagua,Manabí,203.66,7.85,9,8.94,Ciudad no capital,0.26,6.83,5.0
2,Ileana,Antón-Andrés,Sangolquí,Pichincha,14.03,8.75,11,7.28,Ciudad no capital,10.59,7.78,17.7
3,Demetrio,Vazquez,Balzar,Guayas,199.16,8.84,11,5.08,Ciudad no capital,14.09,8.83,34.3
4,Chita del,Giménez,Jaramijó,Manabí,248.22,7.40,9,3.15,Ciudad no capital,7.06,6.72,25.8
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,Alba,Ortega,San Miguel,Bolívar,189.75,8.91,9,6.24,Ciudad no capital,0.97,6.94,5.0
4996,Dr. Jorge,Zelaya,Bahía de Caráquez,Manabí,216.67,8.20,10,6.19,Ciudad no capital,10.85,7.85,17.0
4997,María,López,La Troncal,Cañar,264.16,8.76,12,3.79,Ciudad no capital,20.00,10.00,44.8
4998,Ricardo Luis Arias,Ocampo,Sucúa,Morona Santiago,251.46,8.40,10,8.36,Ciudad no capital,0.49,7.97,5.0


In [34]:
df_gastos

,nombre,apellido,ciudad,becado,tipo_universidad,categoria_gasto,gasto_categoria
0,Raimundo Llopis,Hierro,Latacunga,0,Pública,Vivienda,100.97
1,Raimundo Llopis,Hierro,Latacunga,0,Pública,Alimentación,94.33
2,Raimundo Llopis,Hierro,Latacunga,0,Pública,Transporte,21.80
3,Raimundo Llopis,Hierro,Latacunga,0,Pública,Educación,31.79
4,Raimundo Llopis,Hierro,Latacunga,0,Pública,Ocio y personales,36.29
...,...,...,...,...,...,...,...
29995,Fulgencio de,Miralles,Paute,0,Pública,Alimentación,105.99
29996,Fulgencio de,Miralles,Paute,0,Pública,Transporte,33.81
29997,Fulgencio de,Miralles,Paute,0,Pública,Educación,45.54
29998,Fulgencio de,Miralles,Paute,0,Pública,Ocio y personales,52.65


# Que falta origen General

Agregar semestre (1-12)

Año (yyyy, aleatorio)

Etiqueta periodo (yyyy-A) usar el mismo del año

Carrera (aleatorio)

Genero (aleatorio)

Edad (ajustado para mayor probabilidad entre 18 a 25 menor entre >18 o <25)

Modalidad (90% presencial, el otro 10% las otras)

Zona ( poner todos Quito)

tipo_vivienda ( más porbabilidad arrendada)

sector [ver aqui](https://ecu.postcodebase.com/es/region2/quitopichincha?page=4)

codigo_postal [mismo sitio](https://ecu.postcodebase.com/es/region2/quitopichincha?page=4) (depende del sector)


Estos 2 últimos si se puede sesgar para zonas centrales de  Quito bien, sino aleatorio igual

In [35]:
df_general

,nombre,apellido,ciudad,provincia,longitud,latitud,distancia,tipo_universidad,universidad
0,Raimundo Llopis,Hierro,Latacunga,Cotopaxi,-78.614576,-0.934031,80.19,Pública,Universidad Central del Ecuador (UCE)
1,Benjamín Zoé,Trejo,Tosagua,Manabí,-80.257719,-0.775521,203.66,Pública,Universidad Central del Ecuador (UCE)
2,Ileana,Antón-Andrés,Sangolquí,Pichincha,-78.448348,-0.328882,14.03,Pública,Escuela Politécnica del Ejército (ESPE)
3,Demetrio,Vazquez,Balzar,Guayas,-79.936715,-1.306299,199.16,Pública,Universidad Central del Ecuador (UCE)
4,Chita del,Giménez,Jaramijó,Manabí,-80.613458,-0.974388,248.22,Privada,Universidad Tecnológica Equinoccial (UTE)
...,...,...,...,...,...,...,...,...,...
4995,Alba,Ortega,San Miguel,Bolívar,-79.127262,-1.812057,189.75,Pública,Escuela Politécnica del Ejército (ESPE)
4996,Dr. Jorge,Zelaya,Bahía de Caráquez,Manabí,-80.423572,-0.599965,216.67,Privada,Universidad Tecnológica Equinoccial (UTE)
4997,María,López,La Troncal,Cañar,-79.375163,-2.433689,264.16,Pública,Universidad Andina Simón Bolívar (UASB)
4998,Ricardo Luis Arias,Ocampo,Sucúa,Morona Santiago,-78.172738,-2.456011,251.46,Pública,Universidad Central del Ecuador (UCE)


In [36]:
df_general.to_csv("df_general.csv", index=False)

# **DataFrame general**
agregacion del semestre y periodo comprendido entre el año 2023 y 2025

In [37]:
#Semestre/año y periodo
df_general = pd.read_csv("df_general.csv")

np.random.seed(42)
df_general["semestre"] = np.random.randint(1, 13, size=len(df_general))
df_general["anio"] = np.random.choice([2023, 2024, 2025], size=len(df_general))
df_general["periodo"] = df_general["anio"].astype(str) + "-" + np.random.choice(["A", "B"], size=len(df_general))



Se agrego las carreras de manera aleatoria (+50 carreras), el genero (masculino, femenino, no binario y no definido), la modadlidad de estudio centrado al 90% presencial y el resto online, se definio la zona a "Quito" para todos, el tipo de vivienda se le asigno con mas del 50% de probalidad a cada estudiante

In [38]:
#Carreras, genero, edad, Modalidad, Zona, tipo de vivienda
carreras = [
    "Ingeniería en Sistemas", "Ingeniería Civil", "Ingeniería Mecánica", "Ingeniería Electrónica",
    "Ingeniería Ambiental", "Ingeniería Química", "Ingeniería Biomédica", "Ingeniería Industrial",
    "Ingeniería Automotriz", "Ingeniería Petroquímica", "Ingeniería Agronómica",

    "Medicina", "Odontología", "Enfermería", "Fisioterapia", "Nutrición", "Laboratorio Clínico",

    "Arquitectura", "Diseño Gráfico", "Artes Visuales", "Diseño Industrial", "Diseño de Modas",

    "Economía", "Administración", "Contabilidad", "Finanzas", "Marketing", "Comercio Exterior",
    "Negocios Internacionales",

    "Psicología", "Psicopedagogía", "Psicología Clínica",

    "Derecho", "Criminología", "Trabajo Social",

    "Comunicación Social", "Periodismo", "Publicidad",

    "Educación Inicial", "Educación Básica", "Educación Física",

    "Turismo", "Gastronomía", "Hotelería",

    "Matemáticas", "Física", "Química", "Biología",

    "Tecnologías de la Información", "Ciberseguridad", "Ciencia de Datos"
]

generos = ["Femenino", "Masculino", "No binario", "Prefiere no decir"]
p_generos = [0.48, 0.48, 0.02, 0.02]

edades = np.random.normal(21, 2.5, len(df_general))
edades = np.clip(edades, 17, 45).astype(int)

modalidades = ["Presencial", "En línea", "Híbrida"]
p_modalidad = [0.90, 0.05, 0.05]

df_general["carrera"] = np.random.choice(carreras, size=len(df_general))
df_general["genero"] = np.random.choice(generos, p=p_generos, size=len(df_general))
df_general["edad"] = edades
df_general["modalidad"] = np.random.choice(modalidades, p=p_modalidad, size=len(df_general))
df_general["zona"] = "Quito"

# Tipo de vivienda
tipos_viv = ["Arrendada", "Familiar", "Propia", "Residencia universitaria"]
p_viv = [0.55, 0.25, 0.10, 0.10]
df_general["tipo_vivienda"] = np.random.choice(tipos_viv, p=p_viv, size=len(df_general))



Se carga un archivo excel con los sectores y codigos postales de Quito y se procese a asignar a cada estudiante haciendo un sesgo hacia las zonas mas centricas de las universidades para asignarles mayor probabilidad

In [39]:
# Cargar sectores
sectores = pd.read_excel("CodigosPostalxSector.xlsx")
sectores.columns = ["ciudad", "sector", "codigo_postal"]

# Zonas céntricas
zonas_centricas = [
    "Centro Histórico", "La Mariscal", "La Floresta",
    "La Carolina", "Belisario Quevedo", "Benalcazar",
    "Cotocollao", "Chaupicruz (La Concepción)", "Condado"
]

# Peso
sectores["peso"] = sectores["sector"].apply(lambda x: 3 if x in zonas_centricas else 1)

# Probabilidad normalizada
sectores["prob"] = sectores["peso"] / sectores["peso"].sum()

# Selección
np.random.seed(42)
idx = np.random.choice(sectores.index, size=len(df_general), p=sectores["prob"].values)

# Agregar al df_general
df_general["sector"] = sectores.loc[idx, "sector"].values
df_general["codigo_postal"] = sectores.loc[idx, "codigo_postal"].values



# Visualizacion de los respectivos df

In [40]:
df_general

,nombre,apellido,ciudad,provincia,longitud,latitud,distancia,tipo_universidad,universidad,semestre,anio,periodo,carrera,genero,edad,modalidad,zona,tipo_vivienda,sector,codigo_postal
0,Raimundo Llopis,Hierro,Latacunga,Cotopaxi,-78.614576,-0.934031,80.19,Pública,Universidad Central del Ecuador (UCE),7,2025,2025-B,Nutrición,Femenino,20,Presencial,Quito,Arrendada,Gonzalez Suarez,170107
1,Benjamín Zoé,Trejo,Tosagua,Manabí,-80.257719,-0.775521,203.66,Pública,Universidad Central del Ecuador (UCE),4,2023,2023-B,Gastronomía,Femenino,22,Presencial,Quito,Arrendada,Tababela,170183
2,Ileana,Antón-Andrés,Sangolquí,Pichincha,-78.448348,-0.328882,14.03,Pública,Escuela Politécnica del Ejército (ESPE),11,2025,2025-B,Hotelería,Masculino,24,Presencial,Quito,Familiar,Pifo,170175
3,Demetrio,Vazquez,Balzar,Guayas,-79.936715,-1.306299,199.16,Pública,Universidad Central del Ecuador (UCE),8,2024,2024-B,Contabilidad,Masculino,22,Presencial,Quito,Familiar,La Vicentina,170112
4,Chita del,Giménez,Jaramijó,Manabí,-80.613458,-0.974388,248.22,Privada,Universidad Tecnológica Equinoccial (UTE),5,2023,2023-A,Ingeniería en Sistemas,Masculino,21,Presencial,Quito,Arrendada,Chavezpamba,170158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,Alba,Ortega,San Miguel,Bolívar,-79.127262,-1.812057,189.75,Pública,Escuela Politécnica del Ejército (ESPE),9,2024,2024-A,Ingeniería Civil,Femenino,20,Presencial,Quito,Familiar,San Marcos,170114
4996,Dr. Jorge,Zelaya,Bahía de Caráquez,Manabí,-80.423572,-0.599965,216.67,Privada,Universidad Tecnológica Equinoccial (UTE),1,2025,2025-A,Tecnologías de la Información,Masculino,24,Presencial,Quito,Familiar,Carcelen,170120
4997,María,López,La Troncal,Cañar,-79.375163,-2.433689,264.16,Pública,Universidad Andina Simón Bolívar (UASB),2,2024,2024-B,Ingeniería Química,Femenino,23,Presencial,Quito,Arrendada,El Inca,170124
4998,Ricardo Luis Arias,Ocampo,Sucúa,Morona Santiago,-78.172738,-2.456011,251.46,Pública,Universidad Central del Ecuador (UCE),2,2023,2023-A,Negocios Internacionales,Masculino,19,Presencial,Quito,Residencia universitaria,Quito,170150


In [41]:
df_bienestar

,nombre,apellido,ciudad,provincia,distancia,promedio_académico,viajes_origen,indice_bienestar,origen_bienestar,distancia_universidad,horas_sueño,tiempo_traslado_min
0,Raimundo Llopis,Hierro,Latacunga,Cotopaxi,80.19,8.72,10,8.40,Capital de provincia,0.38,7.60,5.0
1,Benjamín Zoé,Trejo,Tosagua,Manabí,203.66,7.85,9,8.94,Ciudad no capital,0.26,6.83,5.0
2,Ileana,Antón-Andrés,Sangolquí,Pichincha,14.03,8.75,11,7.28,Ciudad no capital,10.59,7.78,17.7
3,Demetrio,Vazquez,Balzar,Guayas,199.16,8.84,11,5.08,Ciudad no capital,14.09,8.83,34.3
4,Chita del,Giménez,Jaramijó,Manabí,248.22,7.40,9,3.15,Ciudad no capital,7.06,6.72,25.8
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,Alba,Ortega,San Miguel,Bolívar,189.75,8.91,9,6.24,Ciudad no capital,0.97,6.94,5.0
4996,Dr. Jorge,Zelaya,Bahía de Caráquez,Manabí,216.67,8.20,10,6.19,Ciudad no capital,10.85,7.85,17.0
4997,María,López,La Troncal,Cañar,264.16,8.76,12,3.79,Ciudad no capital,20.00,10.00,44.8
4998,Ricardo Luis Arias,Ocampo,Sucúa,Morona Santiago,251.46,8.40,10,8.36,Ciudad no capital,0.49,7.97,5.0


In [42]:
df_economia_ingresos

,nombre,apellido,becado,tipo_ingreso,tipo_universidad,ingreso_cantidad
0,Raimundo Llopis,Hierro,0,Apoyo familiar,Pública,400.32
1,Benjamín Zoé,Trejo,0,Apoyo familiar,Pública,341.56
2,Ileana,Antón-Andrés,0,Trabajo,Pública,282.38
3,Demetrio,Vazquez,0,Trabajo,Pública,326.15
4,Demetrio,Vazquez,0,Apoyo familiar,Pública,339.81
...,...,...,...,...,...,...
7161,María,López,0,Apoyo familiar,Pública,418.54
7162,Ricardo Luis Arias,Ocampo,0,Trabajo,Pública,242.30
7163,Ricardo Luis Arias,Ocampo,0,Apoyo familiar,Pública,295.92
7164,Fulgencio de,Miralles,0,Crédito educativo,Pública,220.73


In [43]:
df_gastos

,nombre,apellido,ciudad,becado,tipo_universidad,categoria_gasto,gasto_categoria
0,Raimundo Llopis,Hierro,Latacunga,0,Pública,Vivienda,100.97
1,Raimundo Llopis,Hierro,Latacunga,0,Pública,Alimentación,94.33
2,Raimundo Llopis,Hierro,Latacunga,0,Pública,Transporte,21.80
3,Raimundo Llopis,Hierro,Latacunga,0,Pública,Educación,31.79
4,Raimundo Llopis,Hierro,Latacunga,0,Pública,Ocio y personales,36.29
...,...,...,...,...,...,...,...
29995,Fulgencio de,Miralles,Paute,0,Pública,Alimentación,105.99
29996,Fulgencio de,Miralles,Paute,0,Pública,Transporte,33.81
29997,Fulgencio de,Miralles,Paute,0,Pública,Educación,45.54
29998,Fulgencio de,Miralles,Paute,0,Pública,Ocio y personales,52.65


# Descargar

In [44]:
df_general.to_csv("df_general.csv", index=False)
df_bienestar.to_excel("df_bienestar.xlsx", index= False)

# **Origen Ingresos**

In [45]:
df_economia_ingresos

,nombre,apellido,becado,tipo_ingreso,tipo_universidad,ingreso_cantidad
0,Raimundo Llopis,Hierro,0,Apoyo familiar,Pública,400.32
1,Benjamín Zoé,Trejo,0,Apoyo familiar,Pública,341.56
2,Ileana,Antón-Andrés,0,Trabajo,Pública,282.38
3,Demetrio,Vazquez,0,Trabajo,Pública,326.15
4,Demetrio,Vazquez,0,Apoyo familiar,Pública,339.81
...,...,...,...,...,...,...
7161,María,López,0,Apoyo familiar,Pública,418.54
7162,Ricardo Luis Arias,Ocampo,0,Trabajo,Pública,242.30
7163,Ricardo Luis Arias,Ocampo,0,Apoyo familiar,Pública,295.92
7164,Fulgencio de,Miralles,0,Crédito educativo,Pública,220.73


In [46]:
df_economia_ingresos.to_csv("df_economia_ingresos.csv", index=False)

# **Origen Gastos**

In [47]:
df_gastos

,nombre,apellido,ciudad,becado,tipo_universidad,categoria_gasto,gasto_categoria
0,Raimundo Llopis,Hierro,Latacunga,0,Pública,Vivienda,100.97
1,Raimundo Llopis,Hierro,Latacunga,0,Pública,Alimentación,94.33
2,Raimundo Llopis,Hierro,Latacunga,0,Pública,Transporte,21.80
3,Raimundo Llopis,Hierro,Latacunga,0,Pública,Educación,31.79
4,Raimundo Llopis,Hierro,Latacunga,0,Pública,Ocio y personales,36.29
...,...,...,...,...,...,...,...
29995,Fulgencio de,Miralles,Paute,0,Pública,Alimentación,105.99
29996,Fulgencio de,Miralles,Paute,0,Pública,Transporte,33.81
29997,Fulgencio de,Miralles,Paute,0,Pública,Educación,45.54
29998,Fulgencio de,Miralles,Paute,0,Pública,Ocio y personales,52.65


In [48]:
df_gastos.columns

Index(['nombre', 'apellido', 'ciudad', 'becado', 'tipo_universidad',
       'categoria_gasto', 'gasto_categoria'],
      dtype='object')

In [49]:
df_gastos.to_csv("df_gastos.csv", index=False)

# Agrega errores a los datos
Se va a generar 4 archivos nuevos uno para cada DF con datos, nulos, duplicados, calculos erroneos y formatos erroneos

Cargar dataframes base

In [50]:
df_general = pd.read_csv("df_general.csv")
df_bienestar = pd.read_excel("df_bienestar.xlsx")
df_ingresos = pd.read_csv("df_economia_ingresos.csv")
df_gastos = pd.read_csv("df_gastos.csv")

np.random.seed(42)

Vamos a realizar la mezcla del df_general con los diferentes tipos de errores

Datos mal calculados

In [51]:

# Cargar dataset base
df_general = pd.read_csv("df_general.csv")
np.random.seed(42)

# Insertar 7% de filas completamente vacías
n_blank = max(1, int(len(df_general) * 0.07))
blank_rows = pd.DataFrame([{col: None for col in df_general.columns} for _ in range(n_blank)])
df_temp = pd.concat([df_general, blank_rows], ignore_index=True)
df_general_vacios = df_temp.sample(frac=1, random_state=42).reset_index(drop=True)

# Inyección de nulos por columna (1%–3%)
df = df_general_vacios.copy()
for col in df.columns:
    p = np.random.uniform(0.01, 0.03)
    n = max(1, int(len(df) * p))
    df.loc[np.random.choice(df.index, size=n, replace=False), col] = None
df_general_vacios_columnas = df.copy()

# Duplicados exactos (5% del dataset)
df = df_general_vacios_columnas.copy()
n_dup = max(1, int(len(df) * 0.05))
dup_rows = df.sample(n=n_dup, random_state=99)
df_dup = pd.concat([df, dup_rows], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Inyección de errores de tipo (según data type)
df_err = df_dup.copy()
df_original = pd.read_csv("df_general.csv")
numeric_cols = df_original.select_dtypes(include=["int64", "float64"]).columns
text_cols = [c for c in df_err.columns if c not in numeric_cols]

# Errores en columnas numéricas
for col in numeric_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["texto", "imposible", "formato"])
        if e == "texto": df_err.at[i, col] = np.random.choice(["ERR", "???", "BAD"])
        elif e == "imposible": df_err.at[i, col] = np.random.choice([-9999, 99999, 1e12])
        else: df_err.at[i, col] = f"VAL{np.random.randint(1000,9999)}"

# Errores en columnas textuales
for col in text_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["numero", "corrupto", "especial", "largo"])
        if e == "numero": df_err.at[i, col] = np.random.randint(100000, 999999)
        elif e == "corrupto": df_err.at[i, col] = np.random.choice(["###", "!ERROR!", None])
        elif e == "especial": df_err.at[i, col] = np.random.choice(["∞∞∞", "<script>", "±§¶"])
        else: df_err.at[i, col] = "X" * np.random.randint(30, 120)

# Inconsistencias y errores de cálculo
df_calc = df_err.copy()

if "edad" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=111).index, "edad"] = np.random.choice([150, 0, 300])
if "distancia" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=112).index, "distancia"] = np.random.choice([-50, 20000])
if "latitud" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=113).index, "latitud"] = np.random.choice([200, -200])
if "longitud" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=114).index, "longitud"] = np.random.choice([-9999, 200])
if "semestre" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=115).index, "semestre"] = np.random.choice([0, 50])
if "anio" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=116).index, "anio"] = np.random.choice([1800, "2099X"])
if "periodo" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=117).index, "periodo"] = np.random.choice(["AAAA-B", "XX-XX"])
if "carrera" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=118).index, "carrera"] = np.random.choice(["Carrera???", "NO DEFINIDA"])

# Corrección robusta para evaluar distancia numérica
if "modalidad" in df_calc and "distancia" in df_calc:
    dist = (
        df_calc["distancia"].astype(str)
        .str.replace(r"[^0-9\-\.]", "", regex=True)
        .replace("", np.nan)
        .astype(float)
    )
    far_idx = dist[dist > 200].index
    if len(far_idx) > 0:
        df_calc.loc[np.random.choice(far_idx, size=max(1,int(len(far_idx)*0.30)), replace=False), "modalidad"] = "Presencial"

# Corrupción masiva (5%–10% celdas aleatorias)
df_corrupt = df_calc.copy()
n_corr = max(1, int(df_corrupt.size * np.random.uniform(0.05, 0.10)))
corrupt_vals = ["###CORRUPT###", "@@@", "<NULL>", "INVALID", "BROKEN", "{BAD}"]

for _ in range(n_corr):
    r = np.random.randint(0, df_corrupt.shape[0])
    c = np.random.randint(0, df_corrupt.shape[1])
    df_corrupt.iat[r, c] = np.random.choice(corrupt_vals)

# Mezcla final y exportación
df_final = df_corrupt.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_csv("df_general_errores.csv", index=False)


/tmp/ipython-input-69188573.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_temp = pd.concat([df_general, blank_rows], ignore_index=True)
/tmp/ipython-input-69188573.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'BAD' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if e == "texto": df_err.at[i, col] = np.random.choice(["ERR", "???", "BAD"])
/tmp/ipython-input-69188573.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'VAL5429' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  else

vamos a seguir con el df_bienestar con los diferentes tipos de errores

In [52]:
import pandas as pd
import numpy as np

# Cargar dataset base (XLSX)
df_bien = pd.read_excel("df_bienestar.xlsx")
np.random.seed(42)

# Insertar 7% de filas completamente vacías
n_blank = max(1, int(len(df_bien) * 0.07))
blank_rows = pd.DataFrame([{col: None for col in df_bien.columns} for _ in range(n_blank)])
df_bien_vacios = (
    pd.concat([df_bien, blank_rows], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Inyección de valores nulos por columna (1%–3%)
df = df_bien_vacios.copy()
for col in df.columns:
    p = np.random.uniform(0.01, 0.03)
    idx = np.random.choice(df.index, size=max(1, int(len(df) * p)), replace=False)
    df.loc[idx, col] = None
df_bien_vacios_columnas = df.copy()

# Duplicados exactos (5% del dataset)
df = df_bien_vacios_columnas.copy()
n_dup = max(1, int(len(df) * 0.05))
df_dup = (
    pd.concat([df, df.sample(n=n_dup, random_state=99)], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Identificación de columnas numéricas y textuales
df_err = df_dup.copy()
df_original = pd.read_excel("df_bienestar.xlsx")
numeric_cols = df_original.select_dtypes(include=["int64", "float64"]).columns.tolist()
text_cols = [c for c in df_err.columns if c not in numeric_cols]

# Errores sintéticos en datos numéricos
for col in numeric_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["texto", "imposible", "formato"])
        if e == "texto": df_err.at[i, col] = np.random.choice(["ERR", "XXX", "BAD"])
        elif e == "imposible": df_err.at[i, col] = np.random.choice([-9999, 99999, -888, 1e12])
        else: df_err.at[i, col] = f"VAL{np.random.randint(1000, 9999)}"

# Errores sintéticos en datos textuales
for col in text_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["numero", "corrupto", "especial", "largo"])
        if e == "numero": df_err.at[i, col] = np.random.randint(100000, 999999)
        elif e == "corrupto": df_err.at[i, col] = np.random.choice(["###", "ERR!!", None])
        elif e == "especial": df_err.at[i, col] = np.random.choice(["±§¶", "<script>", "∞∞∞", "áéíóú ñ"])
        else: df_err.at[i, col] = "X" * np.random.randint(30, 120)

# Inconsistencias lógicas en variables críticas
df_calc = df_err.copy()

if "promedio_académico" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=101).index, "promedio_académico"] = np.random.choice([-5, 0, 15, 20])
if "viajes_origen" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=102).index, "viajes_origen"] = np.random.choice([-10, 1000, 999])
if "horas_sueño" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=103).index, "horas_sueño"] = np.random.choice([-3, 0, 20, 30])
if "tiempo_traslado_min" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=104).index, "tiempo_traslado_min"] = np.random.choice([-50, 5000, 9999])
if "indice_bienestar" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=105).index, "indice_bienestar"] = np.random.choice([-10, 0, 15, 30])
if "distancia" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=106).index, "distancia"] = np.random.choice([-300, 0, 20000])
if "latitud" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=107).index, "latitud"] = np.random.choice([200, -200, 999])
if "longitud" in df_calc: df_calc.loc[df_calc.sample(frac=0.02, random_state=108).index, "longitud"] = np.random.choice([200, -200, -9999])

# Corrupción masiva aleatoria (5%–10% de celdas)
df_corrupt = df_calc.copy()
n_cells = max(1, int(df_corrupt.size * np.random.uniform(0.05, 0.10)))
corrupt_vals = ["###CORRUPT###", "@@@", "$$$", "INVALID", "<NULL>", "~~~~", "C0RRUPT3D", "{BAD}", "[DAMAGED]"]

for _ in range(n_cells):
    r = np.random.randint(0, df_corrupt.shape[0])
    c = np.random.randint(0, df_corrupt.shape[1])
    df_corrupt.iat[r, c] = np.random.choice(corrupt_vals)

# Exportación final
df_final = df_corrupt.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_excel("df_bienestar_errores.xlsx", index=False)


/tmp/ipython-input-2701875881.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([df_bien, blank_rows], ignore_index=True)
/tmp/ipython-input-2701875881.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'VAL1223' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  else: df_err.at[i, col] = f"VAL{np.random.randint(1000, 9999)}"
/tmp/ipython-input-2701875881.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'VAL2923' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  else: df_err.at[i, 

vamos a seguir con el df_economia_ingresos con los diferentes tipos de errores

In [53]:
# Cargar dataset base
df_eco = pd.read_csv("df_economia_ingresos.csv")
np.random.seed(42)

# Insertar 7% de filas completamente vacías
n_blank = max(1, int(len(df_eco) * 0.07))
blank_rows = pd.DataFrame([{col: None for col in df_eco.columns} for _ in range(n_blank)])
df_eco_vacios = (
    pd.concat([df_eco, blank_rows], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Inserción de valores nulos por columna (1%–3%)
df = df_eco_vacios.copy()
for col in df.columns:
    p = np.random.uniform(0.01, 0.03)
    idx = np.random.choice(df.index, size=max(1, int(len(df) * p)), replace=False)
    df.loc[idx, col] = None
df_eco_vacios_columnas = df.copy()

# Duplicación exacta del 5% de registros
df = df_eco_vacios_columnas.copy()
n_dup = max(1, int(len(df) * 0.05))
df_dup = (
    pd.concat([df, df.sample(n=n_dup, random_state=99)], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Identificación de columnas numéricas y textuales
df_err = df_dup.copy()
df_original = pd.read_csv("df_economia_ingresos.csv")
numeric_cols = df_original.select_dtypes(include=["int64", "float64"]).columns
text_cols = [c for c in df_err.columns if c not in numeric_cols]

# Generación de errores en campos numéricos
for col in numeric_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["texto", "imposible", "formato"])
        if e == "texto": df_err.at[i, col] = np.random.choice(["ERR", "###", "BAD"])
        elif e == "imposible": df_err.at[i, col] = np.random.choice([-9999, 999999, -888, 1e10])
        else: df_err.at[i, col] = f"VAL{np.random.randint(1000, 9999)}"

# Generación de errores en campos textuales
for col in text_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["numero", "corrupto", "especial", "largo"])
        if e == "numero": df_err.at[i, col] = np.random.randint(100000, 999999)
        elif e == "corrupto": df_err.at[i, col] = np.random.choice(["###", "ERR!!", None])
        elif e == "especial": df_err.at[i, col] = np.random.choice(["∞∞∞", "@@@", "áéíóú ñ"])
        else: df_err.at[i, col] = "X" * np.random.randint(30, 120)

# Inconsistencias lógicas específicas del dataset económico
df_calc = df_err.copy()

if "ingreso_cantidad" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=101).index, "ingreso_cantidad"] = np.random.choice([-500, 0, 99999, 1e6])

if "becado" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=102).index, "becado"] = np.random.choice(["SI", "NO", "X", "??"])

if "tipo_ingreso" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=103).index, "tipo_ingreso"] = np.random.choice(["Ingreso Falso", "???", "1234"])

if "tipo_universidad" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=104).index, "tipo_universidad"] = np.random.choice(["XX", "Public@", "Priv@da"])

# Corrupción masiva aleatoria (5%–10% de celdas)
df_corrupt = df_calc.copy()
n_cells = max(1, int(df_corrupt.size * np.random.uniform(0.05, 0.10)))
corrupt_vals = ["###CORRUPT###", "@@@", "$$$", "<NULL>", "INVALID", "{BAD}", "[DAMAGED]", "C0RRUPT3D", "∞∞∞"]

for _ in range(n_cells):
    r = np.random.randint(0, df_corrupt.shape[0])
    c = np.random.randint(0, df_corrupt.shape[1])
    df_corrupt.iat[r, c] = np.random.choice(corrupt_vals)

# Mezcla final y exportación
df_final = df_corrupt.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_csv("df_economia_ingresos_errores.csv", index=False)


/tmp/ipython-input-2119501269.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([df_eco, blank_rows], ignore_index=True)
/tmp/ipython-input-2119501269.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'VAL6105' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  else: df_err.at[i, col] = f"VAL{np.random.randint(1000, 9999)}"


y por ultimo con el df_gastos con los diferentes tipos de errores

In [54]:
# Cargar dataset base
df_gas = pd.read_csv("df_gastos.csv")
np.random.seed(42)

# Insertar 7% de filas completamente vacías
n_blank = max(1, int(len(df_gas) * 0.07))
blank_rows = pd.DataFrame([{col: None for col in df_gas.columns} for _ in range(n_blank)])
df_gas_vacios = (
    pd.concat([df_gas, blank_rows], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Inyección de valores nulos por columna (1%–3%)
df = df_gas_vacios.copy()
for col in df.columns:
    p = np.random.uniform(0.01, 0.03)
    idx = np.random.choice(df.index, size=max(1, int(len(df) * p)), replace=False)
    df.loc[idx, col] = None
df_gas_vacios_columnas = df.copy()

# Inserción de duplicados exactos (5%)
df = df_gas_vacios_columnas.copy()
n_dup = max(1, int(len(df) * 0.05))
df_dup = (
    pd.concat([df, df.sample(n=n_dup, random_state=99)], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Identificación de columnas numéricas y textuales
df_err = df_dup.copy()
df_original = pd.read_csv("df_gastos.csv")
numeric_cols = df_original.select_dtypes(include=["int64", "float64"]).columns
text_cols = [c for c in df_err.columns if c not in numeric_cols]

# Errores sintéticos en columnas numéricas
for col in numeric_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["texto", "imposible", "formato"])
        if e == "texto": df_err.at[i, col] = np.random.choice(["ERR", "XXX", "BAD"])
        elif e == "imposible": df_err.at[i, col] = np.random.choice([-9999, 99999, -500, 1e12])
        else: df_err.at[i, col] = f"VAL{np.random.randint(1000, 9999)}"

# Errores sintéticos en columnas textuales
for col in text_cols:
    p = np.random.uniform(0.01, 0.04)
    for i in np.random.choice(df_err.index, size=max(1, int(len(df_err) * p)), replace=False):
        e = np.random.choice(["numero", "corrupto", "especial", "largo"])
        if e == "numero": df_err.at[i, col] = np.random.randint(100000, 999999)
        elif e == "corrupto": df_err.at[i, col] = np.random.choice(["###", "ERR!!", None])
        elif e == "especial": df_err.at[i, col] = np.random.choice(["∞∞∞", "±§¶", "@@@", "<script>"])
        else: df_err.at[i, col] = "X" * np.random.randint(30, 120)

# Inconsistencias lógicas específicas del dataset de gastos
df_calc = df_err.copy()

if "gasto_categoria" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=101).index, "gasto_categoria"] = np.random.choice([-500, 0, 100000, 1e6])

if "becado" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=102).index, "becado"] = np.random.choice(["SI", "NO", "X", "??"])

if "categoria_gasto" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=103).index, "categoria_gasto"] = np.random.choice(["Gasto Falso", "???", "1234", "Ninguno", "XXXX"])

if "ciudad" in df_calc:
    df_calc.loc[df_calc.sample(frac=0.02, random_state=104).index, "ciudad"] = np.random.choice(["12345", "Ciudad??", "<UNK>", "∞∞∞"])

# Corrupción masiva (5%–10% de celdas aleatorias)
df_corrupt = df_calc.copy()
n_cells = max(1, int(df_corrupt.size * np.random.uniform(0.05, 0.10)))
corrupt_vals = ["###CORRUPT###", "<NULL>", "$$$", "@@@", "DAMAGED", "[BAD]", "{ERR}", "BROKEN", "∞∞∞", "INVALID"]

for _ in range(n_cells):
    r = np.random.randint(0, df_corrupt.shape[0])
    c = np.random.randint(0, df_corrupt.shape[1])
    df_corrupt.iat[r, c] = np.random.choice(corrupt_vals)

# Exportación final
df_final = df_corrupt.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_csv("df_gastos_errores.csv", index=False)


/tmp/ipython-input-3040642954.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([df_gas, blank_rows], ignore_index=True)
/tmp/ipython-input-3040642954.py:42: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ERR' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if e == "texto": df_err.at[i, col] = np.random.choice(["ERR", "XXX", "BAD"])
